# Notebook 2 SOLUTIONS: Bayesian Quadrature on Alanine Dipeptide

In the previous notebook we used **Bayesian Optimization (BO)** to find
where the force is zero  i.e. the minima/maxima of the free energy surface.

In this notebook we use **Bayesian Quadrature (BUQ)** to do something more
ambitious: reconstruct the **entire free energy surface** A(φ) by integrating
the mean force.

### What is the difference?

| | Bayesian Optimization | Bayesian Quadrature |
|---|---|---|
| **Goal** | Find minimum of f(x) | Compute integral of f(x) |
| **Surrogate** | GP on \|force(φ)\| | GP on force(φ) directly |
| **Acquisition** | Reduce uncertainty near minimum | Reduce uncertainty of integral |
| **Output** | Best φ found | F(φ) + error bars |

### What you will learn
- How BUQ places new simulation points to reduce **integral uncertainty**
- How kernel choice and hyperparameters affect the GP over the force
- How the IVR acquisition function differs from EI/LCB
- How sampling patterns differ between BO (notebook 1) and BUQ (this notebook)

For more information, check out our paper: https://arxiv.org/abs/2601.08783


> **Note:** We use `Adipep1DSystem`, which wraps the same metadynamics data
> as notebook 1, so you can directly compare sampling strategies.

## 0. Imports

In [55]:
import numpy as np
import matplotlib.pyplot as plt

from buq.bq_runner import BQConfig, BayesianQuadratureRunner
from buq.sample_systems.mock import Adipep1DFromGrid

## 1. The system: alanine dipeptide φ

`Adipep1DSystem` wraps the precomputed metadynamics data.
It exposes the mean force f(φ) = dA/dφ via `system.get_force(x)`,
exactly like `Mock1DSystem` does for the analytical parabola.

Let's first look at what the system gives us. Again, this is the same plot as in the first tutorial.

In [57]:
system = Adipep1DFromGrid()

# Evaluate force and FES on a dense grid for reference
phi_grid = np.linspace(system.bounds[0], system.bounds[1], 300)
negative_force = np.array([system.get_force([p])[0] for p in phi_grid])
fes_ref   = system.true_fes(phi_grid)
fes_ref  -= np.min(fes_ref)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(phi_grid, fes_ref, color="orange")
axes[0].set_xlabel("φ (rad)")
axes[0].set_ylabel("A(φ) (kJ/mol)")
axes[0].set_title("Free Energy Surface")

axes[1].plot(phi_grid, negative_force, color="steelblue")
axes[1].axhline(0, color="k", linestyle="--", linewidth=0.8, label="force = 0")
axes[1].set_xlabel("φ (rad)")
axes[1].set_ylabel("dA/dφ (kJ/mol/rad)")
axes[1].set_title("Mean Negative Force")
axes[1].legend()

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory

## 2. Configure BUQ

`BQConfig` controls the GP kernel and the BUQ loop.

Key parameters:
- `kernel_type`: shape of the GP covariance `"Matern12"`, `"Matern32"`, `"Matern52"`, `"RBF"`
- `lengthscale`: how quickly the GP varies with φ, see previous tutorial
- `noise`: assumed observation noise (this is a white kernel)
- `acq_function`: `"IVR"` (Integral Variance Reduction), `"US"` (Uncertainty Sampling), `"MI"` (Mutual Information)
- `grid_size_1d`: resolution of the internal φ grid, for now this is not too important, when dealing with higher dimensions, having a high resolution grid slows down the computation of the integral

### TODO: try changing these settings and see what happens!
- `kernel_type`
- `lengthscale` 
- `noise`

In [ ]:
# --- Settings: change these! ---
kernel_type = "Matern32"   
lengthscale = 2.9          
noise       = 1e-6       
# --------------------------------

config = BQConfig(
    kernel_type=kernel_type,
    lengthscale=lengthscale,
    noise=noise,
    acq_function="IVR",
    grid_size_1d=200,
    use_mini=False,
)

## 3. Initialize the runner

We start with a small set of initial points where the force has already been
evaluated. The runner fits an initial GP to these observations.

### TODO: try different initial point placements
- What happens if all initial points are clustered in one region?

In [ ]:
# --- Settings: change these! ---
initial_points = np.array([[-2.0], [0.0], [2.0]])  # TODO: try different placements
# --------------------------------

runner = BayesianQuadratureRunner(system, config)
runner.initialize(initial_points)

print("Initial points:", initial_points.ravel())

# Plot initial state: FES estimate, acquisition function, and GP over the force
runner.plot_fes(show=True)
runner.plot_acq(weight_var=1.0, weight_fes=0.0, show=True)
runner.plot_derivatives(show=True)

**Questions:**
- Where does the acquisition function suggest sampling next? Why?
- How does the initial FES estimate compare to the reference (dashed line)?

## 4. The BUQ loop

At each step the runner:
1. Evaluates the IVR acquisition function on the φ grid
2. Picks the φ that maximally reduces integral variance
3. Queries the force at that φ (`system.get_force`)
4. Updates the GP posterior and recomputes the FES

The acquisition is a weighted combination:

$$\text{acq}(\phi) = (1-  w_\text{fes}) \cdot \text{IVR}(\phi) - w_\text{fes} \cdot \hat{F}(\phi)$$

- `weight_var=1, weight_fes=0`: pure IVR reduce integral variance uniformly
- `weight_fes > 0`: bias sampling toward high free energy regions (exploitation!)

###  TODO: try changing the weights and number of steps

In [ ]:
# --- Settings: change these! ---
n_steps    = 5     
weight_fes = 0.8  
# --------------------------------
runner = BayesianQuadratureRunner(system, config)
runner.initialize(initial_points)

for i in range(n_steps):
    print(f"\n--- Step {i+1}/{n_steps} ---")
    runner.run_one_query(weight_fes=weight_fes)
    runner.plot_fes(show=True)
    runner.plot_acq(weight_fes=weight_fes, show=True)

**Questions:**
- How do the sampling locations compare to notebook 1 (BO)?
- Does BUQ sample near the force zero crossings, or elsewhere?
- What changes when you increase `weight_fes`?

## 5. Effect of kernel and lengthscale

The kernel controls the GP's assumptions about the smoothness of the force.

- **Small lengthscale**: GP varies quickly → can fit sharp features, but needs
  more points to cover the domain
- **Large lengthscale**: GP varies slowly → smooth FES, may miss sharp features
- **RBF**: infinitely smooth (strong assumption)
- **Matern12/32/52**: less smooth, more realistic for MD forces, where Matern12 is really for sharp, rugged derivatives, while Matern52 is smoother 

### TODO: run each cell and compare the FES reconstructions

In [ ]:
# Helper, no need to change this
def run_buq(kernel_type, lengthscale, noise=1e-6, n_steps=8,
            weight_var=1.0, weight_fes=0.0, title=""):
    """Run a full BUQ loop and return the runner."""
    cfg = BQConfig(
        kernel_type=kernel_type,
        lengthscale=lengthscale,
        noise=noise,
        acq_function="IVR",
        grid_size_1d=200,
        use_mini=False,
    )
    r = BayesianQuadratureRunner(system, cfg)
    r.initialize(np.array([[-2.0], [0.0], [2.0]]))
    for _ in range(n_steps):
        r.run_one_query(weight_var=weight_var, weight_fes=weight_fes)
    print(f"\n{title}")
    r.plot_derivatives(show=True)
    r.plot_fes(show=True)

    return r

In [ ]:
r_short = run_buq("Matern32", lengthscale=0.1, title="Matern32, lengthscale=0.1")

In [ ]:
r_default = run_buq("Matern32", lengthscale=0.75, title="Matern32, lengthscale=0.75")

In [ ]:
# Long lengthscale oversmoothed
r_long = run_buq("Matern32", lengthscale=2.0, title="Matern32, lengthscale=2.0")

In [ ]:
# RBF kernel
r_rbf = run_buq("RBF", lengthscale=0.75, title="RBF, lengthscale=0.75")

**Questions:**
- Which kernel/lengthscale gives the best FES reconstruction?
- With a short lengthscale: does the GP overfit the initial points?
- With a long lengthscale: does the FES miss any features?
- Why might RBF be a poor choice for MD force data?

## 6. Acquisition functions: IVR vs US vs MI

So far we used **IVR** (Integral Variance Reduction), which directly minimizes
the variance of the integral estimate.

Two alternatives are available:
- **US** (Uncertainty Sampling): samples where GP variance is highest
- **MI** (Mutual Information): samples where information gain is highest

IVR is the most principled choice for BUQ, but the others can be useful
for comparison.

### TODO: run each cell and compare sampling patterns

In [ ]:
r_ivr = run_buq("Matern32", lengthscale=0.9, title="IVR")
# Note: to use a different acquisition, change acq_function in BQConfig inside run_buq,
# or copy the helper and modify it:

def run_buq_acq(acq_function, title=""):
    cfg = BQConfig(
        kernel_type="Matern32",
        lengthscale=0.9,
        noise=1e-6,
        acq_function=acq_function,
        grid_size_1d=200,
        use_mini=False,
    )
    r = BayesianQuadratureRunner(system, cfg)
    r.initialize(np.array([[-2.0], [0.0], [2.0]]))
    for _ in range(8):
        r.run_one_query(weight_var=1.0, weight_fes=0.0)
    print(f"\n{title}")
    r.plot_fes(show=True)
    r.plot_acq(weight_var=1.0, weight_fes=0.0, show=True)
    return r

In [ ]:
r_us = run_buq_acq("US", title="Uncertainty Sampling")

In [ ]:
r_mi = run_buq_acq("MI", title="Mutual Information")

**Questions:**
- Do US and MI sample in different locations than IVR?
- Which gives the best FES reconstruction after 8 queries?
- Why is IVR the most natural choice for free energy estimation?

## 7. BUQ vs BO: where do they sample?

Let's directly compare the sampling locations:
- **BO** (notebook 1): minimizes |force(φ)| → clusters near zero crossings
- **BUQ** (this notebook): minimizes integral variance → spreads across domain

The cell below runs a clean BUQ loop and plots the query locations on the FES.

In [ ]:
cfg = BQConfig(
    kernel_type="Matern32", lengthscale=0.9, noise=1e-6,
    acq_function="US", grid_size_1d=200, use_mini=False,
)
runner_compare = BayesianQuadratureRunner(system, cfg)
runner_compare.initialize(np.array([[-2.0], [0.0], [2.0]]))

for _ in range(10):
    runner_compare.run_one_query(weight_var=1.0, weight_fes=0.0)

buq_queries = runner_compare.X_data[:, 0]  # all queried phi values

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(phi_grid, fes_ref, color="orange", label="true FES")
ax.scatter(buq_queries, np.zeros_like(buq_queries) - 1.0,
           marker="|", s=300, color="steelblue", linewidths=2,
           label="BUQ queries", zorder=5)
ax.set_xlabel("φ (rad)")
ax.set_ylabel("F(φ) (kJ/mol)")
ax.set_title("BUQ query locations vs FES")
ax.legend()
plt.tight_layout()
plt.show()

**Key insight:**
- BO clusters queries near the **minimum** of |force| (zero crossing = FES minimum)
- BUQ spreads queries to reduce **integral uncertainty** across the whole domain

Neither is universally better it depends on your goal:
- Want to find the **minimum quickly**? → BO
- Want to reconstruct the **full FES accurately**? → BUQ

## 8. Bonus: convergence of the barrier height

As BUQ adds more points, the FES estimate should converge to the reference.
Let's track the **barrier height** (max of FES) as a function of BUQ iteration.

In [ ]:
cfg = BQConfig(
    kernel_type="Matern32", lengthscale=0.9, noise=1e-6,
    acq_function="IVR", grid_size_1d=200, use_mini=False,
)
runner_conv = BayesianQuadratureRunner(system, cfg)
runner_conv.initialize(np.array([[-2.0], [0.0], [2.0]]))

barrier_estimates = []
n_conv_steps = 15

for i in range(n_conv_steps):
    runner_conv.run_one_query(weight_var=1.0, weight_fes=0.0)
    fes_now = runner_conv.current_fes_1d.copy()
    fes_now -= np.min(fes_now)
    barrier_estimates.append(np.max(fes_now))

barrier_ref = np.max(fes_ref)

plt.figure(figsize=(7, 4))
plt.plot(range(1, n_conv_steps + 1), barrier_estimates, "o-",
         color="steelblue", label="BUQ estimate")
plt.axhline(barrier_ref, color="orange", linestyle="--",
            label="reference (metadynamics)")
plt.xlabel("BUQ iteration")
plt.ylabel("Barrier height (kJ/mol)")
plt.title("Convergence of FES barrier height")
plt.legend()
plt.tight_layout()
plt.show()

**Questions:**
- How many BUQ iterations does it take to converge to the reference barrier height?
- Does changing the lengthscale affect convergence speed?
- How would this compare to random sampling (no acquisition function)?

## Summary

| | Notebook 1: BO | Notebook 2: BUQ |
|---|---|---|
| **Objective** | Find zero of force | Reconstruct full FES |
| **GP models** | \|force(φ)\| | force(φ) directly |
| **Acquisition** | EI / PI / LCB | IVR / US / MI |
| **Sampling** | Clusters near minimum | Spreads across domain |
| **Output** | Best φ found | F(φ) with error bars |

BUQ is the principled approach when you need the **full free energy surface**
and want to know **how uncertain** your estimate is  which is exactly what
matters in enhanced sampling MD.